# Knowledge Base Qualit Assessment

## 1. Introduction

This notebook implements the proposed quality assessment framework for symbolic assembly knowledge bases presented in this work. It demonstrates the application of both the structural quality assessment and the engineering knowledge assessment using a representative symbolic assembly knowledge base. Each quality criterion is evaluated independently according to the assessment methodology defined in the paper, and the resulting observations are combined to provide an overall assessment of the knowledge base quality. The implementation serves as a research prototype to illustrate the applicability of the proposed framework rather than a fully automated industrial assessment tool.

In [960]:
import json
import pandas as pd
from pathlib import Path

performed_test_results={}

## 2. Load Knowledge Base

Load and parse the planning rules

In [961]:
from pathlib import Path
import re

RULESET = Path("ASP_Ruleset.lp")

with open(RULESET, "r", encoding="utf-8") as f:
    lines = f.readlines()

knowledge_base = {}

current_operation = None

for line in lines:

    line = line.rstrip()

    # Detect a new operation rule set
    match = re.match(r"%:?\s*([A-Za-z0-9_]+)_Planning_LP", line)

    if match:
        current_operation = match.group(1)

        knowledge_base[current_operation] = {
            "sections": {},
            "rules": []
        }

        continue

    if current_operation:
        if not line.startswith("%") and len(line)>0:
            knowledge_base[current_operation]["rules"].append(line)
            continue 


print(f"Detected {len(knowledge_base)} operation rule sets.")
operation_rules_in_kb=sum([len(knowledge_base.get(p)["rules"]) for p in knowledge_base])
print(f"{operation_rules_in_kb} Operation Specific Rules")


Detected 10 operation rule sets.
195 Operation Specific Rules


In [962]:
section_pattern = re.compile(r"%\s*(.+?)\s+Planning Rules")

for operation in knowledge_base:

    current_section = None

    knowledge_base[operation]["sections"] = {}

    for line in knowledge_base.get(operation)["rules"]:

        match = section_pattern.match(line)

        if match:

            current_section = match.group(1).strip()

            knowledge_base[operation]["sections"][current_section] = []

            continue

        if current_section:

            line = line.strip()

            if line and not line.startswith("%"):

                knowledge_base[operation]["sections"][current_section].append(line)

print(knowledge_base.get('Screw'))

{'sections': {}, 'rules': ['screw_connection_start(CONN,END) :- screw_connection(CONN), END = 0 .', 'screw_connection_end(CONN,END) :- screw_tighten_end(CONN,END) .', 'conn_component_count(CONN, COUNT) :-COUNT = #count{COMP : has_connected_component(CONN,COMP,ORDER),ORDER!=0},has_connected_component(CONN,_,_).', 'single_component_connection(CONN) :-conn_component_count(CONN,0).', 'screw_assemble_start(CONN,STEP) :- screw_connection_start(CONN,STEP) .', 'screw_assemble_range(CONN,START,COUNT) :- screw_assemble_start(CONN,START),conn_component_count(CONN, COUNT).', 'screw_step_assemble(CONN,STEP) :- screw_assemble_range(CONN,START,COUNT), STEP= START..COUNT, not single_component_connection(CONN).', 'screw_assemble_end(CONN,COUNT+1) :- screw_assemble_start(CONN,STEP), conn_component_count(CONN,COUNT).', 'action(position,CONN,none,COMP,0,0) :- screw_step_assemble(CONN,0), has_connected_component(CONN,COMP,0).', 'action(position,CONN,COMP_A,COMP_B,SUB_STEP,0) :- screw_step_assemble(CONN,SUB

Inspect loaded data

In [963]:
for operation in knowledge_base:

    print(operation)

    for section in knowledge_base[operation]["sections"]:

        print("   ", section)

Screw
Screw_Clamp
Glue
Plugin
Drill
Welding
Documentation
Coat
Pushin
Threading


In [964]:
action_pattern = re.compile(r"^action\(([^,]+)")

for operation in knowledge_base:

    actions = set()

    for rule in knowledge_base[operation]["rules"]:

        match = action_pattern.search(rule)

        if match:

            actions.add(match.group(1))

    knowledge_base[operation]["actions"] = sorted(actions)

print(knowledge_base.get('Screw')["actions"])

['adjust', 'insert', 'insert_pluginnut', 'position', 'position_washer', 'tighten', 'tighten_nut', 'turnon']


## 3. Structural Assessment

### Consistency 

In [965]:
CONSISTENCY_SCHEMA = {

    ("action", 6): {
        "key": [1,2,3,4,5],
        "value": [0]
    },

    ("tool", 2): {
        "key": [1],
        "value": [0]
    },

    ("position_with", 4): {
        "key": [0,1,2],
        "value": [3]
    },

    ("tighten_with", 6): {
        "key": [0,1,2],
        "value": [3,4,5]
    },

    ("insert_with", 5): {
        "key": [0,1,2,3],
        "value": [4]
    },

    ("adjust_with", 4): {
        "key": [0,1,2],
        "value": [3]
    },

    ("turnon_with", 5): {
        "key": [0,1,2,3],
        "value": [4]
    }

}

In [966]:
import re

# -------------------------------------------------------------------
# Regular expressions
# -------------------------------------------------------------------

rule_pattern = re.compile(r"^(.*?)\s*:-\s*(.*?)\.$")
fact_pattern = re.compile(r"^(.*?)\.$")
operation_pattern = re.compile(r"%:?\s*([A-Za-z0-9_]+)_Planning_LP")


# -------------------------------------------------------------------
# Helper: split arguments while respecting nested parentheses
# -------------------------------------------------------------------

def split_arguments(text):
    args = []
    current = ""
    depth = 0

    for char in text:

        if char == "," and depth == 0:
            args.append(current.strip())
            current = ""
            continue

        if char == "(":
            depth += 1

        elif char == ")":
            depth -= 1

        current += char

    if current:
        args.append(current.strip())

    return args


# -------------------------------------------------------------------
# Helper: parse a predicate
# -------------------------------------------------------------------

def parse_atom(atom):

    atom = atom.strip()

    match = re.match(r"(\w+)\((.*)\)", atom)

    if not match:
        return None

    return {
        "predicate": match.group(1),
        "arguments": split_arguments(match.group(2))
    }


# -------------------------------------------------------------------
# Parse the complete ASP file
# -------------------------------------------------------------------

parsed_rules = []

current_operation = "Global"

with open("ASP_Ruleset.lp", "r", encoding="utf-8") as f:

    for line_number, line in enumerate(f, start=1):

        line = line.strip()

        # Ignore empty lines
        if not line:
            continue

        # ------------------------------------------------------------
        # Detect operation block
        # ------------------------------------------------------------

        match = operation_pattern.match(line)

        if match:

            current_operation = match.group(1)

            continue

        # Ignore comments
        if line.startswith("%"):
            continue

        # ------------------------------------------------------------
        # Parse normal rule
        # ------------------------------------------------------------

        rule = rule_pattern.match(line)

        if rule:

            head = parse_atom(rule.group(1))

            body = []

            for atom in split_arguments(rule.group(2)):

                parsed = parse_atom(atom)

                if parsed is not None:
                    body.append(parsed)

            parsed_rules.append({

                "operation": current_operation,

                "head": head,

                "body": body,

                "raw": line,

                "line": line_number

            })

            continue

        # ------------------------------------------------------------
        # Parse fact
        # ------------------------------------------------------------

        fact = fact_pattern.match(line)

        if fact:

            head = parse_atom(fact.group(1))

            if head:

                parsed_rules.append({

                    "operation": current_operation,

                    "head": head,

                    "body": [],

                    "raw": line,

                    "line": line_number

                })


print(f"Parsed {len(parsed_rules)} rules.")


# -------------------------------------------------------------------
# Show some statistics
# -------------------------------------------------------------------

from collections import Counter

operations = Counter()
predicates = Counter()

for rule in parsed_rules:

    operations[rule["operation"]] += 1

    if rule["head"]:

        predicates[rule["head"]["predicate"]] += 1


print("\nRules per operation")
print("-"*40)

for op, count in operations.items():
    print(f"{op:20} {count}")

print("\nPredicates")
print("-"*40)

for pred, count in predicates.most_common():
    print(f"{pred:25} {count}")

Parsed 264 rules.

Rules per operation
----------------------------------------
Global               69
Screw                46
Screw_Clamp          18
Glue                 59
Plugin               10
Drill                8
Welding              15
Documentation        7
Coat                 13
Pushin               12
Threading            7

Predicates
----------------------------------------
action                    65
sub_actions               19
connection                9
optional_info             9
component                 8
is_for                    7
tool                      6
tighten_with              5
glue_step                 5
has_component             3
position_with             3
screw                     2
insert_with               2
screw_insert_start        2
screw_clamp_step          2
glue_stick_on_start       2
weld_step                 2
has_base                  1
connection_point          1
requirement               1
component_connection      1
component_connec

In [967]:
from collections import defaultdict

from collections import defaultdict

def check_consistency(parsed_rules, schema):

    report = []

    for rule in parsed_rules:

        head = rule["head"]

        if head is None:
            continue

        predicate = head["predicate"]
        arity = len(head["arguments"])

        # --------------------------------------------------------
        # Predicate + arity must exist in schema
        # --------------------------------------------------------
        if (predicate, arity) not in schema:
            continue

        definition = schema[(predicate, arity)]

        key_idx = definition["key"]
        value_idx = definition["value"]

        args = head["arguments"]

        key = tuple(args[i] for i in key_idx)
        value = tuple(args[i] for i in value_idx)

        report.append({
            "predicate": predicate,
            "arity": arity,
            "key": key,
            "value": value,
            "rule": rule
        })

    # ------------------------------------------------------------
    # Group by predicate + arity + key
    # ------------------------------------------------------------

    grouped = defaultdict(list)

    for item in report:

        grouped[
            (
                item["predicate"],
                item["arity"],
                item["key"]
            )
        ].append(item)

    # ------------------------------------------------------------
    # Detect conflicts
    # ------------------------------------------------------------

    conflicts = []

    for (predicate, arity, key), entries in grouped.items():

        values = defaultdict(list)

        for entry in entries:
            values[entry["value"]].append(entry["rule"])

        if len(values) > 1:

            conflicts.append({

                "predicate": predicate,

                "arity": arity,

                "key": key,

                "values": values

            })

    return conflicts

In [968]:
conflicts = check_consistency(parsed_rules, CONSISTENCY_SCHEMA)
performed_test_results["Consistency"]={"type":"analysis","result":f"Conflicts found: {len(conflicts)}"}


if not conflicts:
    print("No potential consistency conflicts detected.")

for conflict in conflicts:

    print("=" * 100)
    print(f"Potential Consistency Conflict")
    print("=" * 100)
    print(f"Predicate : {conflict['predicate']}/{conflict['arity']}")
    print(f"Key       : {conflict['key']}")
    print()

    # -------------------------------------------------------------
    # Build body sets
    # -------------------------------------------------------------

    rule_bodies = []

    for value, rules in conflict["values"].items():

        for rule in rules:

            body = {
                f"{atom['predicate']}({', '.join(atom['arguments'])})"
                for atom in rule["body"]
            }

            rule_bodies.append({
                "assignment": value,
                "rule": rule,
                "body": body
            })

    # -------------------------------------------------------------
    # Common conditions
    # -------------------------------------------------------------

    if rule_bodies:
        common_conditions = set.intersection(
            *(entry["body"] for entry in rule_bodies)
        )
    else:
        common_conditions = set()

    print("Common Conditions")
    print("-" * 40)

    if common_conditions:
        for cond in sorted(common_conditions):
            print(f"  ✓ {cond}")
    else:
        print("  None")

    print()

    # -------------------------------------------------------------
    # Differing conditions
    # -------------------------------------------------------------

    print("Differing Conditions")
    print("-" * 40)

    for entry in rule_bodies:

        print()

        assignment = ", ".join(entry["assignment"])

        print(f"Assignment : {assignment}")
        print(f"Operation  : {entry['rule']['operation']}")
        print(f"Line       : {entry['rule']['line']}")

        differing = sorted(entry["body"] - common_conditions)

        if differing:

            for cond in differing:
                print(f"   + {cond}")

        else:
            print("   No differing conditions.")

    print()
    print("Engineer Assessment")
    print("-" * 40)
    print("[ ] Conditions are mutually exclusive")
    print("[ ] Potential consistency conflict")
    print("[ ] Rule modification required")

    print("\n")

Potential Consistency Conflict
Predicate : tighten_with/6
Key       : ('CONN', 'STEP', 'COMP')

Common Conditions
----------------------------------------
  None

Differing Conditions
----------------------------------------

Assignment : ID, SIZE, none
Operation  : Global
Line       : 79
   + action(tighten, CONN, POINT, COMP, STEP)
   + has_head(COMP, SIZE, _)
   + is_for(TOOL, COMP)
   + tool(TOOL, ID)

Assignment : ID, none, none
Operation  : Global
Line       : 80
   + action(tighten, CONN, POINT, COMP, STEP)
   + is_for(TOOL, COMP)
   + tool(TOOL, ID)

Assignment : ID, none, none
Operation  : Global
Line       : 81
   + action(tighten, CONN, POINT, COMP, STEP)
   + has_head(COMP, _, _)
   + is_for("Hand", COMP)
   + tool("Hand", ID)

Assignment : ID, none, none
Operation  : Global
Line       : 82
   + action(tighten_nut, CONN, POINT, COMP, STEP)
   + tool("Socket Wrench", ID)

Assignment : ID, "15", "Nm"
Operation  : Global
Line       : 83
   + action(tighten, CONN, POINT, COMP, 

### Completeness

In [969]:
# ------------------------------------------------------------------
# Operation-specific knowledge
# ------------------------------------------------------------------

OPERATION_SCHEMA = {

    "Screw": ["action"],
    "Screw_Clamp": ["action"],
    "Glue": ["action"],
    "Plugin": ["action"],
    "Drill": ["action"],
    "Welding": ["action"],
    "Documentation": ["action"],
    "Coat": ["action"],
    "Pushin": ["action"],
    "Threading": ["action"],
    "Jointing": ["action"],
    "Sealing": ["action"],
    "Isolate": ["action"]
}


# ------------------------------------------------------------------
# Shared engineering knowledge
# ------------------------------------------------------------------

SHARED_SCHEMA = [

    "tool",
    "position_with",
    "insert_with",
    "adjust_with",
    "turnon_with",
    "tighten_with",
    "optional_info",
    

]

In [970]:
from collections import defaultdict

operation_predicates = defaultdict(set)
global_predicates = set()

for rule in parsed_rules:

    if rule["head"] is None:
        continue

    predicate = rule["head"]["predicate"]

    operation = rule["operation"]

    if operation == "Global":

        global_predicates.add(predicate)

    else:

        operation_predicates[operation].add(predicate)

In [971]:
print("="*100)
print("OPERATION COMPLETENESS")
print("="*100)

for operation, required in OPERATION_SCHEMA.items():

    available = operation_predicates.get(operation, set())

    present = sorted(set(required) & available)

    missing = sorted(set(required) - available)

    score = len(present) / len(required) * 100

    print()
    print("="*70)
    print(f"Operation : {operation}")
    print(f"Completeness : {score:.1f}%")
    print()

    print("Implemented Knowledge")
    print("---------------------")

    for p in present:
        print(f"  ✓ {p}")

    if missing:

        print()
        print("Missing Knowledge")
        print("-----------------")

        for p in missing:
            print(f"  ✗ {p}")

    else:

        print()
        print("Knowledge Base Complete")

OPERATION COMPLETENESS

Operation : Screw
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Screw_Clamp
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Glue
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Plugin
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Drill
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Welding
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Documentation
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge Base Complete

Operation : Coat
Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ action

Knowledge 

In [972]:
print()
print("="*100)
print("SHARED ENGINEERING KNOWLEDGE COMPLETENESS")
print("="*100)

present = sorted(set(SHARED_SCHEMA) & global_predicates)

missing = sorted(set(SHARED_SCHEMA) - global_predicates)

score = len(present) / len(SHARED_SCHEMA) * 100

print()
print(f"Completeness : {score:.1f}%")
print()

print("Implemented Knowledge")
print("---------------------")

for p in present:
    print(f"  ✓ {p}")

if missing:

    print()
    print("Missing Knowledge")
    print("-----------------")

    for p in missing:
        print(f"  ✗ {p}")

else:

    print()
    print("Shared Engineering Knowledge Complete")


performed_test_results["Completeness"]={"type":"test","result":f"{score:.1f} %"}


SHARED ENGINEERING KNOWLEDGE COMPLETENESS

Completeness : 100.0%

Implemented Knowledge
---------------------
  ✓ adjust_with
  ✓ insert_with
  ✓ optional_info
  ✓ position_with
  ✓ tighten_with
  ✓ tool
  ✓ turnon_with

Shared Engineering Knowledge Complete


### Redundancy

In [973]:
from collections import defaultdict

def canonical_atom(atom):
    """
    Convert an atom into a canonical string.
    Example:
    tool(CONN,STEP,"Hand")
    """

    return f"{atom['predicate']}({','.join(atom['arguments'])})"


def canonical_rule(rule):
    """
    Build a canonical representation of a rule.

    Rules are considered identical if they have
    - the same head
    - the same body (independent of literal order)
    """

    head = canonical_atom(rule["head"])

    body = sorted(
        canonical_atom(atom)
        for atom in rule["body"]
    )

    return (
        head,
        tuple(body)
    )


def check_redundancy(parsed_rules):

    rule_index = defaultdict(list)

    for rule in parsed_rules:

        if rule["head"] is None:
            continue

        key = canonical_rule(rule)

        rule_index[key].append(rule)

    redundant = []

    for key, rules in rule_index.items():

        if len(rules) > 1:

            redundant.append({

                "head": key[0],

                "body": key[1],

                "rules": rules

            })

    return redundant

In [974]:
redundant_rules = check_redundancy(parsed_rules)
operation_rules_in_kb=sum([len(knowledge_base.get(p)["rules"]) for p in knowledge_base])
num_redundant_rules=len(redundant_rules)
performed_test_results["Redundancy"]={"type":"test","result":f"{100/operation_rules_in_kb*num_redundant_rules} %"}

if not redundant_rules:

    print("No structurally redundant rules detected.")

else:

    print("="*100)
    print("STRUCTURAL REDUNDANCY ANALYSIS")
    print("="*100)

    for redundancy in redundant_rules:

        print()
        print("="*80)
        print("Redundant Rule")
        print("="*80)

        print()
        print("Head")
        print("-"*40)
        print(redundancy["head"])

        print()
        print("Body")
        print("-"*40)

        if redundancy["body"]:
            for atom in redundancy["body"]:
                print(f"  • {atom}")
        else:
            print("  (fact)")

        print()
        print("Occurrences")
        print("-"*40)

        for rule in redundancy["rules"]:

            print(
                f"[{rule['operation']}] "
                f"Line {rule['line']}"
            )

            print(f"    {rule['raw']}")

        print()

No structurally redundant rules detected.


### Circularity

In [975]:
import networkx as nx

G = nx.DiGraph()

for rule in parsed_rules:

    if rule["head"] is None:
        continue

    head = rule["head"]["predicate"]

    for atom in rule["body"]:

        body = atom["predicate"]

        # Ignore self-dependencies
        if head == body:
            continue

        G.add_edge(head, body)

In [976]:
cycles = list(nx.simple_cycles(G))
performed_test_results["Circularity"]={"type":"analysis","result":f"{cycles} cycles"}

if not cycles:

    print("No circular dependencies detected.")

else:

    print("="*80)
    print("CIRCULARITY ANALYSIS")
    print("="*80)

    for i, cycle in enumerate(cycles,1):

        print()
        print(f"Cycle {i}")

        print(" -> ".join(cycle))

        print(" -> " + cycle[0])

No circular dependencies detected.


### Reachability

In [977]:
import networkx as nx

# ------------------------------------------------------------
# Build dependency graph
#
# body predicate  --->  head predicate
#
# This follows the inference direction.
# ------------------------------------------------------------

G = nx.DiGraph()

for rule in parsed_rules:

    if rule["head"] is None:
        continue

    head = rule["head"]["predicate"]

    # ensure isolated predicates are also present
    G.add_node(head)

    for atom in rule["body"]:

        body = atom["predicate"]

        G.add_edge(body, head)

In [978]:
# ------------------------------------------------------------
# Root predicates
#
# Predicates that are never derived by another rule.
# ------------------------------------------------------------

root_predicates = sorted(
    node
    for node in G.nodes()
    if G.in_degree(node) == 0
)

In [979]:
reachable = set()

for root in root_predicates:

    reachable.add(root)

    reachable.update(
        nx.descendants(G, root)
    )

In [980]:
all_predicates = set(G.nodes())

unreachable = sorted(
    all_predicates - reachable
)

In [981]:
performed_test_results["Reachability"]={"type":"test","result":f"{(100/len(all_predicates)*len(reachable)):.1f} %"}

print("="*100)
print("REACHABILITY ANALYSIS")
print("="*100)

print()
print(f"Predicates analysed           : {len(all_predicates)}")
print(f"Automatically detected roots  : {len(root_predicates)}")
print(f"Reachable predicates          : {len(reachable)}")
print(f"Potentially unreachable       : {len(unreachable)}")

print()

print("Root Predicates")
print("-"*40)

for root in root_predicates:
    print(f"  ✓ {root}")

print()

if not unreachable:

    print("No potentially unreachable predicates detected.")

else:

    print("Potentially Unreachable Predicates")
    print("-"*40)

    for predicate in unreachable:

        print()
        print(predicate)

        print("Rules")

        for rule in parsed_rules:

            if rule["head"] is None:
                continue

            if rule["head"]["predicate"] == predicate:

                print(
                    f"  [{rule['operation']}] Line {rule['line']}"
                )

                print(f"     {rule['raw']}")

REACHABILITY ANALYSIS

Predicates analysed           : 156
Automatically detected roots  : 46
Reachable predicates          : 156
Potentially unreachable       : 0

Root Predicates
----------------------------------------
  ✓ adjust_action
  ✓ assembly_component
  ✓ cable
  ✓ coat_requirement
  ✓ connection_step
  ✓ cut_part
  ✓ default_tool
  ✓ design_part
  ✓ drill_requirement
  ✓ glue_connection
  ✓ glue_type
  ✓ has_connected_component
  ✓ has_connection_point
  ✓ has_drill_point
  ✓ has_head
  ✓ has_hole
  ✓ has_matching
  ✓ has_nut
  ✓ has_pluginnut
  ✓ has_processed_component
  ✓ has_screw
  ✓ has_secure_string
  ✓ has_shape
  ✓ has_threaded_component
  ✓ has_threading_component
  ✓ has_washer
  ✓ has_weld
  ✓ hex_head_screw
  ✓ hex_nut
  ✓ plan_connection
  ✓ plugin_connection
  ✓ push_connection
  ✓ safety_relevant
  ✓ screw_clamp_connection
  ✓ screw_clamp_prepro_end
  ✓ screw_connection
  ✓ screw_point
  ✓ selected_nut_order
  ✓ size
  ✓ spring_washer
  ✓ sticker
  ✓ thread_

## 4. Engineering Assessment

### Domain Coverage

In [982]:
REFERENCE_OPERATIONS = {

    "Screw",
    "Screw_Clamp",
    "Glue",
    "Plugin",
    "Drill",
    "Welding",
    "Coat",
    "Documentation",
    "Pushin",
    "Threading",
    "Jointing",
    "Sealing",
    "Isolate"

}

In [983]:
implemented = set(knowledge_base.keys())

covered_percentage=100/len(REFERENCE_OPERATIONS)*len(implemented & REFERENCE_OPERATIONS)
uncovered_percentage=100/len(REFERENCE_OPERATIONS)*len(REFERENCE_OPERATIONS-implemented)

performed_test_results["Domain Coverage"]={"type":"test","result":f"{covered_percentage:.1f} %"}

print("="*100)
print("DOMAIN COVERAGE ANALYSIS")
print("="*100)

print()
print("Reference Operation Catalogue")
print("-"*100)
print(f"{len(REFERENCE_OPERATIONS)} operations")

print()
print("Implemented Operations")
print("-"*100)
print(f"{len(implemented & REFERENCE_OPERATIONS)} operations")

print()
print("Coverage")
print("-"*100)
print(f"{covered_percentage:.1f} %")

print()
print("Covered Operations")
print("-"*100)

for op in implemented & REFERENCE_OPERATIONS:
    print(f"• {op}")

print()
print("Not Implemented")
print("-"*100)

for op in REFERENCE_OPERATIONS-implemented:
    print(f"• {op} ")




DOMAIN COVERAGE ANALYSIS

Reference Operation Catalogue
----------------------------------------------------------------------------------------------------
13 operations

Implemented Operations
----------------------------------------------------------------------------------------------------
10 operations

Coverage
----------------------------------------------------------------------------------------------------
76.9 %

Covered Operations
----------------------------------------------------------------------------------------------------
• Documentation
• Pushin
• Threading
• Drill
• Plugin
• Glue
• Coat
• Screw
• Welding
• Screw_Clamp

Not Implemented
----------------------------------------------------------------------------------------------------
• Jointing 
• Sealing 
• Isolate 


### Operation Modelling Completeness

In [984]:
OPERATION_MODEL = {

    "Screw": [

        "position",
        "insert",
        "turnon",
        "adjust",
        "tighten"

    ],



    "Glue": [

        "grind",
        "clean",
        "mask",
        "prime",
        "remove_film",
        "cut_to_length",
        "apply_glue",
        "spread_glue",
        "apply_tape",
        "stick_on",
        "remove_excess_glue",
        "remove_tape",
        "smoothen",   

    ],

    "Drill": [

        "drill"

    ],

    
    "Screw_Clamp":[

        "position",
        "adjust",
        "tighten"

    ],

    "Plugin":[

        "position",
        "plugin",
        "secure"

    ],   

    "Welding":[

        "position",
        "weld"

    ],

    "Coat":[
        
        "apply_fhm",
        "position_in_fixture",
        "grind",
        "clean_rough",
        "clean_fine",
        "coat_base",
        "coat_clear"
    
    ],

    "Documentation":[
        
        "document"
        
    ],

    "Pushin":[

        "position",
        "pushin",
        "cut_to_length"
    
    ],

    "Threading":[
        
        "thread"
        
    ],


}

In [985]:
print("="*100)
print("OPERATION MODEL COMPLETENESS ANALYSIS")
print("="*100)


op_coverage=[]
for op in OPERATION_MODEL:
    print()
    print()
    print(f"Operation {op}")
    print("-"*100)

    
    if op in knowledge_base.keys():
        covered_operations=knowledge_base.get(op)["actions"]
        coverage = 100/len(OPERATION_MODEL.get(op))*len(set(OPERATION_MODEL.get(op)) & set(covered_operations))
        op_coverage.append(coverage)

        print()
        print(f"Coverage: {int(coverage)} %")

        print()
        print("Implemented")
        for action in set(OPERATION_MODEL.get(op)) & set(covered_operations):
            print(f"✓ {action}")
        if len(set(OPERATION_MODEL.get(op)) - set(covered_operations))>0:
            print()
            print("Missing")
            for action in set(OPERATION_MODEL.get(op)) - set(covered_operations):
                print(f"✗ {action}")

    else:
        print()
        print(f"Coverage: 0 %")

        print("Missing")
        for action in set(OPERATION_MODEL.get(op)):
            print(f"✗ {action}")


performed_test_results["Operation Modelling Completeness"]={"type":"test","result":f"{(sum(op_coverage)/len(op_coverage)):.1f} %"}

OPERATION MODEL COMPLETENESS ANALYSIS


Operation Screw
----------------------------------------------------------------------------------------------------

Coverage: 100 %

Implemented
✓ turnon
✓ insert
✓ adjust
✓ position
✓ tighten


Operation Glue
----------------------------------------------------------------------------------------------------

Coverage: 100 %

Implemented
✓ cut_to_length
✓ grind
✓ smoothen
✓ clean
✓ prime
✓ remove_excess_glue
✓ remove_film
✓ spread_glue
✓ apply_tape
✓ apply_glue
✓ stick_on
✓ mask
✓ remove_tape


Operation Drill
----------------------------------------------------------------------------------------------------

Coverage: 100 %

Implemented
✓ drill


Operation Screw_Clamp
----------------------------------------------------------------------------------------------------

Coverage: 100 %

Implemented
✓ position
✓ adjust
✓ tighten


Operation Plugin
--------------------------------------------------------------------------------------------------

### Decision Knowledge Validation

In [986]:
DECISION_SCENARIOS = [

    {
        "operation": "Glue",
        "scenario": "Liquid Glue",

        "expected": {
            "grind",
            "clean",
            "mask",
            "prime",
            "apply_glue",
            "spread_glue",
            "remove_excess_glue"
        },

        "forbidden": {
            "remove_film",
            "cut_to_length",
            "apply_tape",
            "smoothen"
        }
    },

    {
        "operation": "Glue",
        "scenario": "Tape",

        "expected": {
            "remove_film",
            "cut_to_length",
            "apply_tape"
        },

        "forbidden": {
            "spread_glue",
            "apply_glue",
            "remove_excess_glue",
            "prime"
        }
    },

    {
        "operation": "Glue",
        "scenario": "Self Adhesive",

        "expected": {
            "remove_film",
        },

        "forbidden": {
            "spread_glue",
            "apply_glue",
            "remove_excess_glue",
            "prime"
        }
    },

        {
        "operation": "Glue",
        "scenario": "Self Adhesive Sticker",

        "expected": {
            "remove_film",
            "smoothen"
        },

        "forbidden": {
            "spread_glue",
            "apply_glue",
            "remove_excess_glue",
            "prime"
        }
    },

    {
        "operation": "Screw",
        "scenario": "With Washer",

        "expected": {
            "position_washer"
        },

        "forbidden": set()
    },

    {
        "operation": "Screw",
        "scenario": "Without Washer",

        "expected": set(),

        "forbidden": {
            "position_washer"
        }
    },

    {
        "operation": "Screw",
        "scenario": "With Nut",

        "expected": {
            "tighten_nut"
        },

        "forbidden": set()
    },

    {
        "operation": "Screw",
        "scenario": "Without Nut",

        "expected": set(),

        "forbidden": {
            "tighten_nut"
        },
    },

    {
        "operation": "Screw",
        "scenario": "With Pluginnut",

        "expected": {
            "insert_pluginnut"
        },

        "forbidden": set()
    },

    {
        "operation": "Screw",
        "scenario": "Without Pluginnut",

        "expected": set(),

        "forbidden": {
            "insert_pluginnut"
        }
    },

    
    {
        "operation": "Screw",
        "scenario": "With Longhole",

        "expected": {
            "adjust"
        },

        "forbidden": set()
    },

    
    {
        "operation": "Screw",
        "scenario": "Without Longhole",

        "expected": set(),

        "forbidden": {
            "adjust"
        }
    },


]

In [987]:
from dataclasses import dataclass


@dataclass
class DecisionScenario:

    operation: str
    scenario: str
    facts: list[str]

In [988]:

def parse_decision_test_cases(filename):

    scenarios = []

    current_operation = None
    current_scenario = None
    current_facts = []

    with open(filename, "r") as f:

        for line in f:

            line = line.strip()

            # ignore empty lines
            if not line:
                continue

            # operation header
            if line.startswith("% Operation: "):
                
                value = line.replace("% Operation:", "").strip()
                
                if current_operation is None:
                    current_operation = value
                    continue

                else:
                    scenarios.append(

                        DecisionScenario(
                            operation=current_operation,
                            scenario=current_scenario,
                            facts=current_facts
                        )

                    )
                
                current_operation = value
                #reset other values
                current_scenario = None
                current_facts = []


                continue

            if line.startswith("% Scenario: "):

                value = line.replace("% Scenario:", "").strip()

                current_scenario = value

                continue

            # ignore other comments
            if line.startswith("%"):
                continue

            current_facts.append(line)

    # append final scenario
    scenarios.append(

        DecisionScenario(
            operation=current_operation,
            scenario=current_scenario,
            facts=current_facts
        )

    )

    return scenarios

In [989]:
import clingo

generated = set()
decision_schema={}

def get_decision_schema(operation, scenario):

    for s in DECISION_SCENARIOS:

        if (s["operation"] == operation and
            s["scenario"] == scenario):

            return s

    return None



test_scenarios=parse_decision_test_cases("Decision_Knowledge_Test_Input.lp")
test_senarios_passed=0

for scenario in test_scenarios:
    
    decision_schema=get_decision_schema(scenario.operation,scenario.scenario)

    generated = set()

    def on_model(model):

        for atom in model.symbols(shown=True):

            if atom.name == "action":
                generated.add(str(atom.arguments[0]))
    
    ctl = clingo.Control(["--models", str(1)])
    ctl.load("ASP_Ruleset.lp")
    for r in scenario.facts:
        ctl.add("base", [], r)
    ctl.ground([("base", [])])

    resutl = ctl.solve(on_model=on_model)

    if decision_schema:
        expected = decision_schema["expected"]
        forbidden = decision_schema["forbidden"]

        missing = expected - generated

        unexpected = generated & forbidden

        passed = (
            len(missing) == 0 and
            len(unexpected) == 0
        )

        print(f"Operation : {scenario.operation}")
        print(f"Scenario  : {scenario.scenario}")
        print()

        if passed:
            print("✓ PASS")
            test_senarios_passed+=1

        else:

            print("✗ FAIL")

            if missing:

                print("\nMissing decisions:")

                for a in sorted(missing):
                    print(f"  ✗ {a}")

            if unexpected:

                print("\nUnexpected decisions:")

                for a in sorted(unexpected):
                    print(f"  ✗ {a}")

        print()
        print("="*100)
        print()

   
performed_test_results["Decision Knowledge Validation"]={"type":"test","result":f"{(100/len(test_scenarios)*test_senarios_passed):.1f} %"}



Operation : Screw
Scenario  : With Washer

✓ PASS


Operation : Screw
Scenario  : Without Washer

✓ PASS


Operation : Screw
Scenario  : With Nut

✓ PASS


Operation : Screw
Scenario  : Without Nut

✓ PASS


Operation : Screw
Scenario  : With Pluginnut

✓ PASS


Operation : Screw
Scenario  : Without Pluginnut

✓ PASS


Operation : Screw
Scenario  : With Longhole

✗ FAIL

Missing decisions:
  ✗ adjust


Operation : Glue
Scenario  : Liquid Glue

✓ PASS


Operation : Glue
Scenario  : Tape

✓ PASS


Operation : Glue
Scenario  : Self Adhesive

✓ PASS


Operation : Glue
Scenario  : Self Adhesive Sticker

✓ PASS




### Engineering Rule Coverage

In [990]:
ENGINEERING_REQUIREMENTS = [

    {
        "id": "ER1",
        "description": "Safety-relevant threaded connections require documentation and torque-controlled tightening.",

        "facts": [
            "safety_relevant(conn,screw)"
        ],

        "expected": [
            ("tighten_with", "Nm"),
            ("action", "document")
        ]
    }

]

In [991]:
from dataclasses import dataclass

@dataclass
class EngineeringRuleScenario:

    rule: str
    description: str
    facts: list[str]

In [992]:

def parse_decision_test_cases(filename):

    scenarios = []

    current_rule = None
    current_description = None
    current_facts = []

    with open(filename, "r") as f:

        for line in f:

            line = line.strip()

            # ignore empty lines
            if not line:
                continue

            # operation header
            if line.startswith("% Rule: "):
                
                value = line.replace("% Rule:", "").strip()
                
                if current_rule is None:
                    current_rule = value
                    continue

                else:
                    scenarios.append(

                        EngineeringRuleScenario(
                            rule=current_rule,
                            description=current_description,
                            facts=current_facts
                        )

                    )
                
                current_rule = value
                #reset other values
                current_description = None
                current_facts = []


                continue

            if line.startswith("% Description: "):

                value = line.replace("% Description:", "").strip()

                current_description = value

                continue

            # ignore other comments
            if line.startswith("%"):
                continue

            current_facts.append(line)

    # append final scenario
    scenarios.append(

        EngineeringRuleScenario(
            rule=current_rule,
            description=current_description,
            facts=current_facts
        )

    )

    return scenarios

In [993]:
import clingo
from collections import defaultdict

generated = defaultdict(list)

def get_decision_schema(rule, description):

    for s in ENGINEERING_REQUIREMENTS:

        if (s["id"] == rule and
            s["description"] == description):

            return s
    return None

#load test cases
test_scenarios=parse_decision_test_cases("Engineering_Rule_Coverage_Test_Input.lp")
test_senarios_passed=0

#run all test cases
for scenario in test_scenarios:

    def on_model(model):

        for atom in model.symbols(shown=True):
            generated[str(atom.name)].append(atom.arguments)
    
    ctl = clingo.Control(["--models", str(1)])
    ctl.load("ASP_Ruleset.lp")
    for r in scenario.facts:
        ctl.add("base", [], r)
    ctl.ground([("base", [])])

    #solve test case
    resutl = ctl.solve(on_model=on_model)

    #load test case schema
    decision_schema=get_decision_schema(scenario.rule,scenario.description)
    if decision_schema:
        facts = decision_schema["facts"]
        expected = decision_schema["expected"]
        
        missing=[]

        for e in expected: 
            found = any(
                any(str(arg).strip("\"") == e[1] for arg in args)
                for args in generated[e[0]]
            )

            if not found:
                missing.append(e)
        
        passed=True
        if len(missing)>0:
            passed=False

        print(f"Rule : {scenario.rule}")
        print(f"Description  : {scenario.description}")
        print()

        if passed:
            print("✓ PASS")
            test_senarios_passed+=1

        else:

            print("✗ FAIL")

            if missing:

                print("\nMissing expected facts:")

                for a in sorted(missing):
                    print(f"  ✗    predicate: {a[0]}\n       parameter: {a[1]}")


        print()
        print("="*100)
        print()

performed_test_results["Engineering Rule Coverage"]={"type":"test","result":f"{(100/len(test_scenarios)*test_senarios_passed):.1f} %"}


Rule : ER1
Description  : Safety-relevant threaded connections require documentation and torque-controlled tightening.

✓ PASS




## 5. Summary

In [995]:

performed_tests={}
performed_tests["StructuralAssessment"]=["Consistency","Completeness","Redundancy","Circularity","Reachability"]
performed_tests["EngineeringAssessment"]=["Domain Coverage","Operation Modelling Completeness","Decision Knowledge Validation","Engineering Rule Coverage"]

summary=[]
failed_tests=[]

print("="*100)
print("KNOWLEGE BASE QUALITY ASSESSMENT")
print("="*100)

print()
print("Structural Assessment")
print("-"*100)

for test in performed_tests["StructuralAssessment"]:
    result=performed_test_results.get(test)["result"]

    if performed_test_results.get(test)["type"]=="test":

        summary.append(str(result)=="100.0 %")
        if str(result)!="100.0 %":
            failed_tests.append(test)

    print(f"{test}:     {result}")


print()
print("Engineering Assessment")
print("-"*100)

for test in performed_tests["EngineeringAssessment"]:
    result=performed_test_results.get(test)["result"]

    if performed_test_results.get(test)["type"]=="test":
        
        summary.append(str(result)=="100.0 %")
        if str(result)!="100.0 %":
            failed_tests.append(test)

    print(f"{test}:     {result}")

print()
print("="*100)
print("Overall Knowledge Base Assessment")

overall_result=not any(s==False for s in summary)
if overall_result:
    print("✓ PASS")

else:
    print("✗ FAIL")

    print()
    print("Failed assessment criteria: ")
    for test in failed_tests:
        print(f"• {test}")

    print()
    print("The knowledge base requires revision before it can be considered complete")
    print("according to the proposed assessment framework.")

    print("="*100)


KNOWLEGE BASE QUALITY ASSESSMENT

Structural Assessment
----------------------------------------------------------------------------------------------------
Consistency:     Conflicts found: 14
Completeness:     100.0 %
Redundancy:     0.0 %
Circularity:     [] cycles
Reachability:     100.0 %

Engineering Assessment
----------------------------------------------------------------------------------------------------
Domain Coverage:     76.9 %
Operation Modelling Completeness:     96.7 %
Decision Knowledge Validation:     90.9 %
Engineering Rule Coverage:     100.0 %

Overall Knowledge Base Assessment
✗ FAIL

Failed assessment criteria: 
• Redundancy
• Domain Coverage
• Operation Modelling Completeness
• Decision Knowledge Validation

The knowledge base requires revision before it can be considered complete
according to the proposed assessment framework.
